# Part 2: Gates and measurement

Practice notebook for *Qubit to Classifier*. Run the examples, fill in the exercises, and compare with the solutions at the end. Every cell needs only Python 3 and NumPy, except one optional Qiskit exercise in chapter 4.1.

## 2.1 Gates are rotations

### The standard gates, and identities you can check

Each gate is a small NumPy array, and the rotation gates are functions of an angle built from the formula R<sub>n</sub>(θ) = cos(θ/2) I − i sin(θ/2) (n<sub>x</sub> X + n<sub>y</sub> Y + n<sub>z</sub> Z), where n is the unit rotation axis. Several identities from the chapter are then checked numerically.

In [ ]:
import numpy as np

I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
S = np.diag([1, 1j])
T = np.diag([1, np.exp(1j * np.pi / 4)])


def rx(theta):
    return np.cos(theta / 2) * I - 1j * np.sin(theta / 2) * X


def ry(theta):
    return np.cos(theta / 2) * I - 1j * np.sin(theta / 2) * Y


def rz(theta):
    return np.cos(theta / 2) * I - 1j * np.sin(theta / 2) * Z


print("HZH = X:        ", np.allclose(H @ Z @ H, X))
print("T T = S:        ", np.allclose(T @ T, S))
print("S X S^dagger = Y:", np.allclose(S @ X @ S.conj().T, Y))
print("Rx(2 pi) = -I:  ", np.allclose(rx(2 * np.pi), -I))
print("Ry(pi/2)|0> =", (ry(np.pi / 2) @ [1, 0]).real.round(4), "which is |+>")

### Exercise 2.1.1: Rz and P differ by a global phase

Check that Rz(λ) = e<sup>−iλ/2</sup> P(λ) for λ = 0.7, where P(λ) = diag(1, e<sup>iλ</sup>). Then find the phase factor between S and Rz(π/2).

<details><summary>Hint</summary>

If A = cB for a number c, then c = A[0, 0] / B[0, 0]. `np.angle(c)` gives its angle.

</details>

In [ ]:
# Your code here

### Exercise 2.1.2: Eight T gates make a full turn

Start from |+⟩ and apply T eight times. After each gate, print the Bloch vector and its angle around the z axis. Each step should turn the arrow 45° about z.

<details><summary>Hint</summary>

The angle around z is `np.degrees(np.arctan2(y, x))`.

</details>

In [ ]:
# Your code here

## 2.2 Measurement and bases

### Measuring along any axis

Most hardware measures only Z, so to measure along X it applies H first, and along Y it applies S† and then H. Reading 0 afterwards means the + outcome along that axis: |0⟩ for Z, |+⟩ for X and |+i⟩ for Y. The code compares those probabilities with the formula (1 + r·n)/2 for the state with θ = π/3 and φ = 0.

In [ ]:
import numpy as np

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
Sdg = np.diag([1, -1j])                       # S dagger
theta = np.pi / 3
psi = np.array([np.cos(theta / 2), np.sin(theta / 2)])
r = np.array([np.sin(theta), 0, np.cos(theta)])   # its Bloch vector

rotate_first = {"Z": np.eye(2), "X": H, "Y": H @ Sdg}   # Y: S dagger first, then H
axis = {"Z": [0, 0, 1], "X": [1, 0, 0], "Y": [0, 1, 0]}
plus = {"Z": "0", "X": "+", "Y": "+i"}                  # the + outcome along each axis
for basis, U in rotate_first.items():
    p_plus = abs((U @ psi)[0]) ** 2               # probability of reading 0
    formula = (1 + r @ axis[basis]) / 2
    label = f"P({plus[basis]})"
    print(f"{basis} basis:  {label:5s} = {p_plus:.3f}   (1 + r.n)/2 = {formula:.3f}")

The numbers match the worked example "one tilted state, measured three ways": 0.75, 0.933 and 0.5.

### Exercise 2.2.1: An expectation value with an error bar

A state has P(0) = 0.62. Simulate 1,000 shots, score each outcome as +1 for 0 and −1 for 1, and print the estimate of ⟨Z⟩ with its standard error √((1 − ⟨Z⟩²)/N). Is the true value 0.24 inside two standard errors?

<details><summary>Hint</summary>

`1 - 2 * outcomes` turns outcomes 0 and 1 into the scores +1 and −1.

</details>

In [ ]:
# Your code here

### Exercise 2.2.2: Tomography of a hidden state

Hide a random state. For N shots per axis, estimate x, y and z by measuring along X, Y and Z, and print how far the estimated Bloch vector is from the true one for N = 100, 1,000 and 10,000.

<details><summary>Hint</summary>

Along an axis, P(+) = (1 + coordinate)/2. Draw the number of + results with `rng.binomial(N, p_plus)` and turn it back into an estimate with 2 × (count / N) − 1.

</details>

In [ ]:
# Your code here

---

# Solutions

### Solution 2.1.1: Rz and P differ by a global phase

In [ ]:
import numpy as np


def rz(lam):
    return np.diag([np.exp(-1j * lam / 2), np.exp(1j * lam / 2)])


def p(lam):
    return np.diag([1, np.exp(1j * lam)])


lam = 0.7
print("Rz = e^(-i lam/2) P:", np.allclose(rz(lam), np.exp(-1j * lam / 2) * p(lam)))
S = p(np.pi / 2)
c = S[0, 0] / rz(np.pi / 2)[0, 0]
print("S = c Rz(pi/2) with c =", np.round(c, 4), " angle of c / pi =", np.angle(c) / np.pi)
print("check:", np.allclose(S, c * rz(np.pi / 2)))

Expected output:

```
Rz = e^(-i lam/2) P: True
S = c Rz(pi/2) with c = (0.7071+0.7071j)  angle of c / pi = 0.25
check: True
```

The ratio is e<sup>iπ/4</sup>, a global phase. On one qubit S and Rz(π/2) are the same gate; controlled, they are not (chapter 3.2).

### Solution 2.1.2: Eight T gates make a full turn

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
T = np.diag([1, np.exp(1j * np.pi / 4)])


def bloch(psi):
    return np.array([np.vdot(psi, P @ psi).real for P in (X, Y, Z)])


psi = np.array([1, 1]) / np.sqrt(2)
for k in range(1, 9):
    psi = T @ psi
    x, y, z = bloch(psi).round(9) + 0      # round away tiny errors such as -1e-17
    print(f"after {k} T: r = ({x:+.3f}, {y:+.3f}, {z:+.3f})   angle = {np.degrees(np.arctan2(y, x)):6.1f} deg")

Expected output:

```
after 1 T: r = (+0.707, +0.707, +0.000)   angle =   45.0 deg
after 2 T: r = (+0.000, +1.000, +0.000)   angle =   90.0 deg
after 3 T: r = (-0.707, +0.707, +0.000)   angle =  135.0 deg
after 4 T: r = (-1.000, +0.000, +0.000)   angle =  180.0 deg
after 5 T: r = (-0.707, -0.707, +0.000)   angle = -135.0 deg
after 6 T: r = (+0.000, -1.000, +0.000)   angle =  -90.0 deg
after 7 T: r = (+0.707, -0.707, +0.000)   angle =  -45.0 deg
after 8 T: r = (+1.000, +0.000, +0.000)   angle =    0.0 deg
```

`arctan2` reports angles between −180° and 180°, so 225° appears as −135°. After eight steps the arrow is back at +x: 8 × 45° = 360°.

### Solution 2.2.1: An expectation value with an error bar

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=6)
N, p0 = 1000, 0.62
outcomes = rng.choice([0, 1], size=N, p=[p0, 1 - p0])
scores = 1 - 2 * outcomes                    # 0 -> +1, 1 -> -1
estimate = scores.mean()
error = np.sqrt((1 - estimate ** 2) / N)
print(f"<Z> is about {estimate:.3f} ± {error:.3f}")
print("true value 0.24 within two standard errors:", abs(estimate - 0.24) < 2 * error)

Expected output:

```
<Z> is about 0.258 ± 0.031
true value 0.24 within two standard errors: True
```

### Solution 2.2.2: Tomography of a hidden state

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=7)
theta, phi = rng.uniform(0, np.pi), rng.uniform(0, 2 * np.pi)
r_true = np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])
print("hidden Bloch vector:", r_true.round(3))

for N in [100, 1000, 10_000]:
    plus_counts = rng.binomial(N, (1 + r_true) / 2)     # one count per axis
    r_est = 2 * plus_counts / N - 1
    print(f"N = {N:6d}: estimate {r_est.round(3)}   error {np.linalg.norm(r_est - r_true):.3f}")

Expected output:

```
hidden Bloch vector: [ 0.738 -0.556 -0.383]
N =    100: estimate [ 0.68 -0.62 -0.38]   error 0.086
N =   1000: estimate [ 0.77  -0.546 -0.388]   error 0.034
N =  10000: estimate [ 0.736 -0.559 -0.38 ]   error 0.004
```

The typical error is about √(2/N): 0.14 for N = 100 and 0.014 for N = 10,000. A single run can land closer or farther; this one happens to land closer than typical at every N.